# Apriori Experiment Notebook

Notebook ini untuk eksperimen parameter Apriori (support/confidence/lift), evaluasi rule, dan plotting hasil.


In [ ]:
from pathlib import Path
import sys
import csv
from itertools import product
from time import perf_counter
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "apriori.py").exists():
            return p
    raise FileNotFoundError("Tidak menemukan project root (apriori.py).")

ROOT = find_project_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from apriori import apriori, load_transactions_from_csv

print("Project root:", ROOT)


## 1) Pilih Dataset

- Default membaca file lokal `data/sample_transactions.csv`.
- Untuk data riil, ganti `DATA_PATH` ke file CSV milikmu.


In [ ]:
DATA_PATH = ROOT / "data" / "sample_transactions.csv"

# Contoh ganti data riil:
# DATA_PATH = Path(r"D:\\Koding\\dataset\\peminjaman_riil.csv")

transaction_id_col = "transaction_id"
item_col = "item"

transactions = load_transactions_from_csv(
    DATA_PATH,
    transaction_id_col=transaction_id_col,
    item_col=item_col,
)
print(f"Dataset: {DATA_PATH}")
print(f"Jumlah transaksi: {len(transactions)}")


## 2) Jalankan Apriori (Single Run)


In [ ]:
min_support = 0.30
min_confidence = 0.60
min_lift = 1.00

frequent_itemsets, rules = apriori(
    transactions=transactions,
    min_support=min_support,
    min_confidence=min_confidence,
    min_lift=min_lift,
)

print("Frequent itemsets:", len(frequent_itemsets))
print("Association rules:", len(rules))

top_n = 15
for i, r in enumerate(rules[:top_n], start=1):
    print(
        f"{i:02d}. IF {list(r.antecedent)} THEN {list(r.consequent)} "
        f"| support={r.support:.4f} confidence={r.confidence:.4f} lift={r.lift:.4f}"
    )


## 3) Eksperimen Parameter (Grid Search)


In [ ]:
supports = [0.2, 0.3, 0.4]
confidences = [0.5, 0.6, 0.7]
exp_min_lift = 1.0

results = []
for s, c in product(supports, confidences):
    t0 = perf_counter()
    _, exp_rules = apriori(
        transactions=transactions,
        min_support=s,
        min_confidence=c,
        min_lift=exp_min_lift,
    )
    runtime_ms = (perf_counter() - t0) * 1000
    row = {
        "min_support": s,
        "min_confidence": c,
        "rules_count": len(exp_rules),
        "max_lift": max((r.lift for r in exp_rules), default=0.0),
        "avg_lift": (sum(r.lift for r in exp_rules) / len(exp_rules)) if exp_rules else 0.0,
        "runtime_ms": runtime_ms,
    }
    results.append(row)

for r in results:
    print(
        f"s={r['min_support']:.2f}, c={r['min_confidence']:.2f} "
        f"-> rules={r['rules_count']}, max_lift={r['max_lift']:.4f}, runtime={r['runtime_ms']:.2f}ms"
    )


In [ ]:
labels = [f"s={r['min_support']}, c={r['min_confidence']}" for r in results]
rules_count = [r["rules_count"] for r in results]
runtime_ms = [r["runtime_ms"] for r in results]

plt.figure(figsize=(12, 4))
plt.bar(labels, rules_count)
plt.xticks(rotation=45, ha="right")
plt.title("Jumlah Rules per Kombinasi Parameter")
plt.ylabel("Rules Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(labels, runtime_ms, marker="o")
plt.xticks(rotation=45, ha="right")
plt.title("Runtime Apriori per Kombinasi Parameter")
plt.ylabel("Runtime (ms)")
plt.tight_layout()
plt.show()


## 4) Simpan Ringkasan Eksperimen


In [ ]:
out_dir = ROOT / "outputs" / "notebook_experiment"
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "experiment_summary.csv"

with out_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["min_support", "min_confidence", "rules_count", "max_lift", "avg_lift", "runtime_ms"],
    )
    writer.writeheader()
    writer.writerows(results)

print("Tersimpan:", out_csv)
